# AutoMechInterp — CPU Judge & Human Review
Select a **CPU runtime**. This notebook scores the same report/layer/agent/S-EAP evidence shown to humans, using `z-ai/glm-5.2:free` through OpenRouter. It needs no model weights and no A100.

Add your key to **Colab Secrets → OPENROUTER_API_KEY**, and enable notebook access. The key is read only into the environment; it is never written into the notebook, ZIP, request logs, reports or score files. The free model can impose daily/request limits. Each score is saved to Drive; rerun later to continue. No automatic paid/model fallback is used.

## Mount Drive and upload the release
Mount Drive so every completed behavior survives a Colab reset. Upload **MIR_colab_bundle.zip** when prompted. The previous `MIR_persisted` directory is historical evidence and is not used to mark new jobs complete.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, sys, json, zipfile, subprocess, hashlib

drive.mount('/content/drive')
BUNDLE = Path('/content/MIR_colab_bundle.zip')
if not BUNDLE.exists():
    uploaded = files.upload()
    candidates = [Path(name) for name in uploaded if name.endswith('.zip')]
    if len(candidates) != 1:
        raise ValueError('Upload exactly one release ZIP')
    BUNDLE = candidates[0]
WORK = Path('/content/MIR_release') / hashlib.sha256(BUNDLE.read_bytes()).hexdigest()[:16]
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for info in archive.infolist():
        target = (WORK / info.filename).resolve()
        if not target.is_relative_to(WORK.resolve()):
            raise ValueError('Unsafe ZIP path')
    manifest = json.loads(archive.read('RELEASE_MANIFEST.json'))
    for name, expected in manifest['files'].items():
        if hashlib.sha256(archive.read(name)).hexdigest() != expected:
            raise ValueError('Release checksum mismatch: ' + name)
    archive.extractall(WORK)
REPO = WORK / 'Automatic-Mechanistic-Research'
PERSIST_TOP = Path('/content/drive/MyDrive/MIR_runs_v3')
PERSIST_TOP.mkdir(parents=True, exist_ok=True)
print('Code:', REPO, '\nDurable results:', PERSIST_TOP)


## Install the tested Python dependencies
This isolated environment reuses Colab's CUDA PyTorch instead of replacing it with a CPU build. A nonzero install/test exit stops execution. The code does not require OpenRouter credentials for GPU extraction.

In [ ]:
VENV = Path('/content/mir_venv')
subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)], check=True)
PYTHON = str(VENV / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '--quiet', '-r', str(REPO/'requirements-colab.txt')], check=True)
os.environ['MPLCONFIGDIR'] = '/content/mir_matplotlib'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
subprocess.run([PYTHON, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO, check=True)


## Select the persisted run and load the secret
Use the run ID printed by the GPU notebook. If exactly one run is present it is selected automatically; otherwise enter the intended ID below. Review outputs are generated from the checksummed jobs.

In [ ]:
from google.colab import userdata
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
RUN_ID = ''  # Paste the GPU notebook's run ID here if there are multiple runs.
if not RUN_ID:
    runs = sorted(p.parent for p in PERSIST_TOP.glob('*/run_manifest.json'))
    if len(runs) != 1:
        raise ValueError('Set RUN_ID explicitly; available runs: ' + ', '.join(p.name for p in runs))
    RUN_ID = runs[0].name
RUN_DIR = PERSIST_TOP / RUN_ID
subprocess.run([PYTHON, 'run_review.py', '--run-dir', str(RUN_DIR), '--export'], cwd=REPO, check=True)
subprocess.run([PYTHON, '-m','llm_review.run_judge', '--preflight'], cwd=REPO, check=True)


## Score a resumable batch
`MAX_ITEMS=50` caps this invocation, not the total study. Increase only within your account limits. `SAMPLE='all'` covers report/layer/agent/S-EAP items. After complete extraction, use `calibration` for the 100-report rubric calibration set, then freeze the prompt/rubric and use `validation` for the disjoint 550-report test set.

The validator does not declare trust from LLM self-scores. Until paired human scoring passes the validation gates, judge health is provisional.

In [ ]:
MAX_ITEMS = 50
SAMPLE = 'all'  # 'calibration', 'validation', or 'all'
ITEM_TYPES = 'report,layer,agent,tool,seap'  # For report validation use 'report'.
subprocess.run([PYTHON, '-m','llm_review.run_judge', '--run-dir', str(RUN_DIR), '--max-items', str(MAX_ITEMS), '--sample', SAMPLE, '--item-types', ITEM_TYPES], cwd=REPO, check=True)


## Human scoring
Copy `review/human_calibration_BLANK_TEMPLATE.csv` and `review/human_validation_BLANK_TEMPLATE.csv` into working files. Two reviewers score independently and fill `reviewer_id` with distinct names; an explicit adjudicated reference uses `reviewer_id=consensus`. Preserve the original independent ratings and the `rubric_version` / `job_sha256` columns; stale evidence or rubric scores are rejected. Use `human_scores_BLANK_TEMPLATE.csv` to review every report, layer, agent, tool and S-EAP element, including explicit not-run rows. Never edit the BLANK_TEMPLATE files because exporting regenerates them.

Use **all eight integer 1–10 scores** per item. Blank cells remain unscored. `item_id` identifies the exact evidence. Whole reports are in `reports/`; for a particular layer/agent element, use the optional cell below to render its exact evidence. Human reviewers must not see judge scores before submitting their own.

The calibration sample has 100 reports (four per angle). The held-out validation sample has 550 reports (one per model–angle cell). These become frozen only after the complete set is available. The main statistical gate requires macro quadratic weighted kappa ≥0.80, bootstrap lower bound ≥0.75, within-one-point agreement ≥80%, and critical-metric kappa ≥0.75. Exact agreement is reported separately.

In [ ]:
ITEM_ID = ''  # Optional: paste one item_id from the human CSV.
if ITEM_ID:
    subprocess.run([PYTHON, 'run_review.py', '--run-dir', str(RUN_DIR), '--item-id', ITEM_ID], cwd=REPO, check=True)
    from IPython.display import Markdown, display
    display(Markdown((RUN_DIR/'review/selected_item.md').read_text()))


## Compare humans with the judge and generate health graphs
Place the completed CSV files under `review/completed_human_scores/`. Include the two independent reviewers and the explicit consensus reference. You may run this cell before any scores exist: it will correctly show **unvalidated / incomplete**, not invented agreement or a pass.

In [ ]:
score_dir = RUN_DIR/'review/completed_human_scores'
score_dir.mkdir(exist_ok=True)
HUMAN_FILES = sorted(score_dir.glob('*.csv'))
cmd = [PYTHON, 'run_review.py', '--run-dir', str(RUN_DIR)]
if HUMAN_FILES: cmd += ['--human-scores', *map(str, HUMAN_FILES)]
subprocess.run(cmd, cwd=REPO, check=True)
print((RUN_DIR/'review/judge_validation.json').read_text())
from IPython.display import display, Image
for p in sorted((RUN_DIR/'review/graphs').glob('0[5-79]*.png')):
    display(Image(filename=str(p)))
